In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

import pickle


In [19]:
df = pd.read_csv("Training.csv")
df.head()


,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis,Unnamed: 133
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN


In [20]:
df.drop("Unnamed: 133", axis=1, inplace=True)


In [21]:
df.shape


(4920, 133)

In [22]:
df.isnull().sum().sum()


0

In [23]:
le = LabelEncoder()
df["prognosis"] = le.fit_transform(df["prognosis"])


In [24]:
X = df.drop("prognosis", axis=1)
y = df["prognosis"]


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [26]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)
print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))


Decision Tree Accuracy: 1.0


In [27]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))


Random Forest Accuracy: 1.0


In [28]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))


Random Forest Accuracy: 1.0


In [29]:
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)
print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))


Naive Bayes Accuracy: 1.0


In [30]:
def evaluate_model(y_true, y_pred):
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average="weighted"))
    print("Recall   :", recall_score(y_true, y_pred, average="weighted"))
    print("F1 Score :", f1_score(y_true, y_pred, average="weighted"))


In [31]:
evaluate_model(y_test, rf_pred)


Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0


In [32]:
pickle.dump(rf_model, open("disease_model.pkl", "wb"))
pickle.dump(le, open("label_encoder.pkl", "wb"))

symptom_list = X.columns.tolist()
pickle.dump(symptom_list, open("symptom_list.pkl", "wb"))


In [33]:
test_df = pd.read_csv("Testing.csv")
test_df.head()


,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,0,0,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Allergy
2,0,0,0,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,GERD
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Chronic cholestasis
4,1,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,Drug Reaction


In [ ]:
test_df.drop("Unnamed: 133", axis=1, inplace=True)

test_X = test_df.drop("prognosis", axis=1)
test_y = test_df["prognosis"]


KeyError: "['Unnamed: 133'] not found in axis"

In [ ]:
test_predictions = rf_model.predict(test_X)
test_predictions[:10]


In [ ]:
def predict_disease(selected_symptoms):
    model = pickle.load(open("disease_model.pkl", "rb"))
    le = pickle.load(open("label_encoder.pkl", "rb"))
    symptoms = pickle.load(open("symptom_list.pkl", "rb"))

    input_data = np.zeros(len(symptoms))

    for symptom in selected_symptoms:
        if symptom in symptoms:
            idx = symptoms.index(symptom)
            input_data[idx] = 1

    input_data = input_data.reshape(1, -1)

    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data).max()

    disease = le.inverse_transform([prediction])[0]

    return disease, round(probability * 100, 2)


In [ ]:
sample_symptoms = ["itching", "skin_rash", "fatigue"]

disease, confidence = predict_disease(sample_symptoms)

print("Predicted Disease :", disease)
print("Confidence        :", confidence, "%")
